# Can we predict `passed_quiz`?
Follows `checklist.md`. Design decisions:
- **Split:** no session date exists, so there is no "later rows" ordering. `student_key` repeats (~7 rows per student), so the test set and every CV fold hold *whole students* (checklist 1, 2).
- **Features:** `enrol_year`, `school_code` (categorical), `prior_score`, `session_minutes`, `hints_used`. `student_key` is an identifier and is dropped. `session_minutes` and `hints_used` are recorded during the session, before the closing quiz, so they are assumed available at prediction time (prediction made at the end of the session, before the quiz). `prior_score` is a placement test; it varies only slightly across a student's rows (SD about 0.3) and that deviation was checked to be unrelated to the outcome.
- **Metric:** ROC AUC for model choice; the decision threshold is chosen on out-of-fold predictions to maximise balanced accuracy.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
df = pd.read_csv('data.csv')
df.shape, df.passed_quiz.mean(), df.student_key.nunique()

## 1. Hold out the test set (whole students) — before any EDA or modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert set(train.student_key).isdisjoint(test.student_key)
print(len(train), len(test), train.passed_quiz.mean(), test.passed_quiz.mean())

## 2. EDA on training rows only

In [ ]:
num = ['enrol_year','prior_score','session_minutes','hints_used']
print(train[num+['passed_quiz']].corr()['passed_quiz'])
print(train.groupby('passed_quiz')[num].mean())
print(train.school_code.nunique(), 'schools')

## 3. Candidate models (named before any scoring)
1. `baseline_prior` — predicts the training pass rate (DummyClassifier).
2. `logreg_C0.1`, `logreg_C1` — logistic regression, standardised numerics + one-hot school.
3. `hgb_shallow`, `hgb_deep` — gradient boosting (max_depth 2 / 4, lr 0.05, 200 iter, no early stopping), numeric features only.
4. `logreg_no_school` — logistic regression C=1 on `prior_score`, `session_minutes`, `hints_used` only.

All preprocessing lives inside each pipeline, so it is fitted inside each CV fold (checklist 3).

In [ ]:
FEATS = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
N = ['enrol_year','prior_score','session_minutes','hints_used']
N3 = ['prior_score','session_minutes','hints_used']
def pre(num, cat=(), scale=True):
    t = [('n', StandardScaler() if scale else 'passthrough', num)]
    if cat: t.append(('c', OneHotEncoder(handle_unknown='ignore'), list(cat)))
    return ColumnTransformer(t)
hgb = lambda d: HistGradientBoostingClassifier(max_depth=d, learning_rate=0.05, max_iter=200, early_stopping=False, random_state=0)
models = {
 'baseline_prior': (Pipeline([('p', pre(N)), ('m', DummyClassifier(strategy='prior'))]), N),
 'logreg_C0.1': (Pipeline([('p', pre(N,['school_code'])), ('m', LogisticRegression(C=0.1, max_iter=2000))]), FEATS),
 'logreg_C1': (Pipeline([('p', pre(N,['school_code'])), ('m', LogisticRegression(C=1, max_iter=2000))]), FEATS),
 'hgb_shallow': (Pipeline([('p', pre(N,scale=False)), ('m', hgb(2))]), N),
 'hgb_deep': (Pipeline([('p', pre(N,scale=False)), ('m', hgb(4))]), N),
 'logreg_no_school': (Pipeline([('p', pre(N3)), ('m', LogisticRegression(C=1, max_iter=2000))]), N3),
}

## 4. Grouped cross-validation on training rows (whole students per fold)

In [ ]:
gkf = GroupKFold(n_splits=5)
y, g = train.passed_quiz.values, train.student_key.values
folds = list(gkf.split(train, y, g))
oof, rows = {}, []
for name,(pipe,cols) in models.items():
    o = np.zeros(len(train)); aucs=[]
    for a,b in folds:
        m = clone(pipe).fit(train.iloc[a][cols], y[a])
        o[b] = m.predict_proba(train.iloc[b][cols])[:,1]
        aucs.append(roc_auc_score(y[b], o[b]))
    oof[name]=o; rows.append((name, np.mean(aucs), np.std(aucs)))
cv = pd.DataFrame(rows, columns=['model','cv_auc_mean','cv_auc_sd']).sort_values('cv_auc_mean', ascending=False)
cv

## 5. Choose model and threshold on CV only

In [ ]:
best = cv.iloc[0].model
ths = np.linspace(0.05, 0.95, 91)
bal = [balanced_accuracy_score(y, oof[best] >= t) for t in ths]
thr = float(ths[int(np.argmax(bal))])
print('CHOSEN MODEL:', best, '| threshold:', round(thr,2), '| OOF balanced acc:', round(max(bal),4))

**Choice recorded here, before the test set is scored:** the model and threshold printed above (highest mean grouped-CV AUC; threshold maximising out-of-fold balanced accuracy). Nothing below changes them.

## 6. Refit on all training rows; score the test set once

In [ ]:
pipe, cols = models[best]
final = clone(pipe).fit(train[cols], y)
p = final.predict_proba(test[cols])[:,1]
yt = test.passed_quiz.values
res = {'test_auc': roc_auc_score(yt, p),
       'test_balanced_acc': balanced_accuracy_score(yt, p >= thr),
       'test_accuracy': accuracy_score(yt, p >= thr),
       'majority_class_accuracy': max(yt.mean(), 1-yt.mean())}
rng = np.random.default_rng(0)
ts = test.student_key.values; ids = np.unique(ts); grp = {s:np.where(ts==s)[0] for s in ids}
bs=[]
for _ in range(500):
    ix = np.concatenate([grp[s] for s in rng.choice(ids, len(ids))])
    bs.append(roc_auc_score(yt[ix], p[ix]))
res['test_auc_95ci_student_bootstrap'] = tuple(np.percentile(bs,[2.5,97.5]).round(3))
res

## 7. Conclusion
**Headline (test set, scored once, 492 held-out students / 2,452 sessions):** the chosen model, logistic regression on `prior_score`, `session_minutes`, `hints_used` at threshold 0.41 (both picked on grouped CV only), reaches **AUC 0.592 (95% student-bootstrap CI 0.569–0.616)** and balanced accuracy 0.569. CV AUC was 0.604, consistent with the test result.

**Interpretation:** there is real but weak signal. Longer sessions and fewer hints go with passing; `prior_score`, `enrol_year` and `school_code` add essentially nothing, and the school one-hot and deeper boosted trees only overfit. At the balanced-accuracy threshold, plain accuracy (0.565) is *below* the majority-class rate (0.582), so the model is not useful for hard pass/fail calls; it is only useful as a weak ranking signal.

**Caveats:** no session date exists, so the split is by student, not by time; if the model will score future sessions of already-known students the estimate may differ. `session_minutes` and `hints_used` are assumed known before the quiz. The two leakage-test-selection candidates from `waldean check` point at the CV cell, where model and threshold are chosen on out-of-fold predictions, not on test rows, so they are false positives.